# CSO distributions of the fine-tuning dataset

This notebook visualizes the distributions of the CSO flows over the two weirs in the SWMM simulations used to fine-tune the metamodel (training, validation and testing splits in `saved_objects/saved_splitted_datasets`).

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.graph_objs as go

import ipywidgets as widgets
from IPython.display import display, clear_output

from extract_specific_timeseries import aggregate_variable_from_multiple_simulations

In [ ]:
INTERVENTION_LEVELS = [1, 5, 10, 20, 50]
indexes = {
    "training":[f"{i:02d}" for i in range(10)],
    "validation":[f"{i:02d}" for i in range(10, 15)],
    "testing":[f"{i:02d}" for i in range(15, 20)]
}
datasplit_type = 'training'

In [ ]:

main_col = "link_weir_30064_40004_Flow_rate"      # 40004 main weir
secondary_col = "link_weir_30065_40003_Flow_rate" # 40003 secondary weir

flowrates_main_weir, flowrates_secondary_weir = aggregate_variable_from_multiple_simulations(
    datasplit_type=datasplit_type,
    indexes=indexes,
    intervention_levels=INTERVENTION_LEVELS,
    base_root=Path("../saved_objects/saved_splitted_datasets"),
    main_col=main_col,
    secondary_col=secondary_col,
    variable="flow_rate"
)

## Example level histograms

In [ ]:


# Data sources available in your notebook
datasets = {
    "Main weir (40004)": flowrates_main_weir,
    "Secondary weir (40003)": flowrates_secondary_weir,
}

# Widgets
dataset_dd = widgets.Dropdown(
    options=list(datasets.keys()),
    value="Main weir (40004)",
    description="Dataset:",
    layout=widgets.Layout(width="380px"),
)

columns_sm = widgets.SelectMultiple(
    options=list(datasets[dataset_dd.value].columns),
    value=tuple(list(datasets[dataset_dd.value].columns)[:1]),
    description="Columns:",
    layout=widgets.Layout(width="380px", height="200px"),
)

bins_slider = widgets.IntSlider(value=50, min=10, max=200, step=5, description="Bins:")
density_cb = widgets.Checkbox(value=False, description="Density")
output = widgets.Output()

def _update_columns(change):
    df = datasets[dataset_dd.value]
    cols = list(df.columns)
    columns_sm.options = cols
    columns_sm.value = tuple(cols[:1]) if cols else tuple()

dataset_dd.observe(_update_columns, names="value")

def _plot(*_):
    with output:
        clear_output(wait=True)

        df = datasets[dataset_dd.value]
        selected_cols = list(columns_sm.value)

        if not selected_cols:
            print("Select at least one column.")
            return

        fig = go.Figure()
        for col in selected_cols:
            fig.add_trace(
                go.Histogram(
                    x=df[col].dropna(),
                    name=col,
                    nbinsx=bins_slider.value,
                    histnorm="probability density" if density_cb.value else None,
                    opacity=0.65,
                )
            )

        fig.update_layout(
            title=f"Histogram - {dataset_dd.value}",
            xaxis_title="Flow rate",
            yaxis_title="Density" if density_cb.value else "Count",
            barmode="overlay",
            template="plotly_white",
            legend_title="Run ID",
            xaxis_range=[-0.01, 2.0],
        )
        fig.show()

# Trigger updates
columns_sm.observe(_plot, names="value")
bins_slider.observe(_plot, names="value")
density_cb.observe(_plot, names="value")
dataset_dd.observe(_plot, names="value")

display(widgets.VBox([dataset_dd, columns_sm, bins_slider, density_cb, output]))
_plot()

## Intervention level histograms

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objs as go

def values_for_level(df: pd.DataFrame, level: int):
    cols = [c for c in df.columns if c.split("_")[-1] == str(level)]
    if not cols:
        return pd.Series(dtype=float)
    vals = pd.concat([df[c] for c in cols], ignore_index=True).dropna()
    return vals

levels = INTERVENTION_LEVELS  # [1, 5, 10, 20, 50]

fig = make_subplots(
    rows=2,
    cols=len(levels),
    subplot_titles=(
        [f"Main weir - {lvl}%" for lvl in levels] +
        [f"Sec. weir - {lvl}%" for lvl in levels]
    ),
)

for i, lvl in enumerate(levels, start=1):
    main_vals = values_for_level(flowrates_main_weir, lvl)
    sec_vals = values_for_level(flowrates_secondary_weir, lvl)

    fig.add_trace(
        go.Histogram(x=main_vals, nbinsx=80, marker_color="#1f77b4", showlegend=False),
        row=1, col=i
    )
    fig.add_trace(
        go.Histogram(x=sec_vals, nbinsx=80, marker_color="#ff7f0e", showlegend=False),
        row=2, col=i
    )

fig.update_layout(
    template="plotly_white",
    title="Aggregated CSO flow-rate histograms by intervention level",
    height=700,
)

for i in range(1, len(levels) + 1):
    fig.update_xaxes(title_text="Flow rate (cms)", range=[0, 1.5], row=1, col=i)
    fig.update_xaxes(title_text="Flow rate (cms)", range=[0, 1.5], row=2, col=i)
    fig.update_yaxes(title_text="", row=1, col=i)
    fig.update_yaxes(title_text="", row=2, col=i)

fig.show()

## Dataset level histograms

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objs as go

# Aggregate all columns into one 1D array per weir
main_vals = flowrates_main_weir.to_numpy().ravel()
main_vals = main_vals[~pd.isna(main_vals)]

secondary_vals = flowrates_secondary_weir.to_numpy().ravel()
secondary_vals = secondary_vals[~pd.isna(secondary_vals)]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Main weir (40004) - aggregated", "Secondary weir (40003) - aggregated")
)

fig.add_trace(
    go.Histogram(x=main_vals, nbinsx=80, name="Main weir", marker_color="#1f77b4"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=secondary_vals, nbinsx=80, name="Secondary weir", marker_color="#ff7f0e"),
    row=1, col=2
)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    title="Aggregated CSO flow-rate distributions"
)

fig.update_xaxes(title_text="Flow rate", range=[0, 1.5], row=1, col=1)
fig.update_xaxes(title_text="Flow rate", range=[0, 1.5], row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.show()

## Total CSO volume

In [ ]:
from scipy.integrate import simpson

TIME_STEP_SECONDS = 60  # seconds

def total_cso_volume_simpson(df: pd.DataFrame, dt_seconds: float = 60.0) -> pd.Series:
    def _integrate_col(col: pd.Series) -> float:
        y = pd.to_numeric(col, errors="coerce").dropna().to_numpy()
        if y.size < 2:
            return 0.0
        return float(simpson(y, dx=dt_seconds))  # m^3

    return df.apply(_integrate_col, axis=0)

# Per-column total CSO volume (m^3)
total_volume_main_m3 = total_cso_volume_simpson(flowrates_main_weir, dt_seconds=TIME_STEP_SECONDS)
total_volume_secondary_m3 = total_cso_volume_simpson(flowrates_secondary_weir, dt_seconds=TIME_STEP_SECONDS)

# Convert m^3 -> 10^6 litres (ML): divide by 1000
total_volume_main_ml = total_volume_main_m3 / 1000.0
total_volume_secondary_ml = total_volume_secondary_m3 / 1000.0

total_cso_volumes = pd.DataFrame({
    "main_weir_40004_ML": total_volume_main_ml,
    "secondary_weir_40003_ML": total_volume_secondary_ml,
})

display(total_cso_volumes)

In [ ]:
# ...existing code...
from plotly.subplots import make_subplots
import plotly.graph_objs as go

n_bins = 5
levels = INTERVENTION_LEVELS  # [1, 5, 10, 20, 50]
level_colors = {
    1: "#1f77b4",
    5: "#2ca02c",
    10: "#ff7f0e",
    20: "#d62728",
    50: "#9467bd",
}

# Extract intervention level from run id index (e.g., "03_20" -> 20)
run_ids = total_cso_volumes.index.astype(str)
run_levels = run_ids.str.split("_").str[-1].astype(int)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Main weir (40004) total volume [ML]",
        "Secondary weir (40003) total volume [ML]",
    ),
)

for lvl in levels:
    mask = run_levels == lvl

    fig.add_trace(
        go.Histogram(
            x=total_cso_volumes.loc[mask, "main_weir_40004_ML"].dropna(),
            nbinsx=n_bins,
            name=f"Level {lvl}",
            legendgroup=f"lvl_{lvl}",
            marker_color=level_colors.get(lvl, "#7f7f7f"),
            opacity=0.65,
            showlegend=True,
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Histogram(
            x=total_cso_volumes.loc[mask, "secondary_weir_40003_ML"].dropna(),
            nbinsx=n_bins,
            name=f"Level {lvl}",
            legendgroup=f"lvl_{lvl}",
            marker_color=level_colors.get(lvl, "#7f7f7f"),
            opacity=0.65,
            showlegend=False,  # avoid duplicate legend entries
        ),
        row=1, col=2
    )

fig.update_layout(
    template="plotly_white",
    title="Distribution of total CSO volume per run (colored by intervention level)",
    height=420,
    barmode="overlay",
)

fig.update_xaxes(title_text="Total volume [ML]", row=1, col=1)
fig.update_xaxes(title_text="Total volume [ML]", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.show()
# ...existing code...